# EBM Internals - Quantile Regression

This notebook covers quantile regression using pinball loss with Explainable Boosting Machines.

Standard regression models (e.g. with RMSE) predict the conditional mean of the target. Quantile regression instead predicts a specific quantile (e.g. median, 10th percentile, 90th percentile). This is useful for:

- **Prediction intervals**: Fit models at the 10th and 90th percentiles to get an 80% prediction interval.
- **Asymmetric risk**: When over-predicting and under-predicting have different costs.
- **Robustness**: Median regression (alpha=0.5) is more robust to outliers than mean regression.

EBMs support quantile regression via the `"quantile"` objective, which uses the pinball loss (also called the quantile loss). The pinball loss for quantile alpha is:

$$L(y, \hat{y}) = \begin{cases} \alpha \cdot (y - \hat{y}) & \text{if } y \geq \hat{y} \\ (1 - \alpha) \cdot (\hat{y} - y) & \text{if } y < \hat{y} \end{cases}$$

This loss penalizes under-predictions by a factor of alpha and over-predictions by a factor of (1 - alpha), causing the model to learn the alpha-quantile of the conditional distribution.

In [ ]:
# boilerplate
from interpret import show
from interpret.glassbox import ExplainableBoostingRegressor
import numpy as np

## Median Regression (alpha=0.5)

Let's start with median regression. With alpha=0.5, the pinball loss penalizes over- and under-predictions equally, so the model learns the conditional median rather than the conditional mean.

In [ ]:
# make a dataset composed of a nominal categorical, and a continuous feature
X = [["Peru", 7.0], ["Fiji", 8.0], ["Peru", 9.0]]
y = [450.0, 550.0, 350.0]

# Fit a quantile EBM for the median (alpha=0.5)
# Eliminate the validation set to handle the small dataset
ebm_median = ExplainableBoostingRegressor(
    objective="quantile:alpha=0.5",
    interactions=0,
    validation_size=0, outer_bags=1, min_samples_leaf=1, min_hessian=1e-9)
ebm_median.fit(X, y)
show(ebm_median.explain_global())

The model structure is identical to a standard regression EBM: an intercept plus additive score contributions from each feature, looked up via binning. The only difference is the loss function used during training.

In [ ]:
print("Intercept:", ebm_median.intercept_)
print("Feature types:", ebm_median.feature_types_in_)
print("Bins:", ebm_median.bins_)
print("Categorical scores:", ebm_median.term_scores_[0])
print("Continuous scores:", ebm_median.term_scores_[1])

Predictions are computed identically to standard regression EBMs: start from the intercept and add lookup table scores for each feature.

In [ ]:
print("Median predictions:", ebm_median.predict(X))
print("Original y values: ", y)

## Prediction Intervals

A key use case for quantile regression is constructing prediction intervals. By fitting separate models at different quantiles, we can estimate the range within which future observations are likely to fall.

Let's use a larger, noisier dataset to demonstrate this. We'll fit models at the 10th, 50th, and 90th percentiles to get an 80% prediction interval.

In [ ]:
from sklearn.datasets import make_regression

X_train, y_train = make_regression(
    n_samples=1000, n_features=5, noise=20.0, random_state=42)

X_test, y_test = make_regression(
    n_samples=200, n_features=5, noise=20.0, random_state=123)

# Fit quantile models at the 10th, 50th, and 90th percentiles
ebm_10 = ExplainableBoostingRegressor(objective="quantile:alpha=0.1")
ebm_50 = ExplainableBoostingRegressor(objective="quantile:alpha=0.5")
ebm_90 = ExplainableBoostingRegressor(objective="quantile:alpha=0.9")

ebm_10.fit(X_train, y_train)
ebm_50.fit(X_train, y_train)
ebm_90.fit(X_train, y_train)

pred_10 = ebm_10.predict(X_test)
pred_50 = ebm_50.predict(X_test)
pred_90 = ebm_90.predict(X_test)

print("First 5 test samples:")
print("  10th percentile:", np.round(pred_10[:5], 2))
print("  50th percentile:", np.round(pred_50[:5], 2))
print("  90th percentile:", np.round(pred_90[:5], 2))
print("  Actual y:       ", np.round(y_test[:5], 2))

In [ ]:
# Verify quantile ordering: q10 < q50 < q90 for most predictions
print("Fraction where q10 < q50:", np.mean(pred_10 < pred_50))
print("Fraction where q50 < q90:", np.mean(pred_50 < pred_90))

# Check empirical coverage of the 80% prediction interval [q10, q90]
coverage = np.mean((y_test >= pred_10) & (y_test <= pred_90))
print(f"Empirical coverage of [q10, q90] interval: {coverage:.1%} (target: ~80%)")

## Visualizing the Prediction Interval

Let's visualize the prediction interval on a sorted subset of test samples.

In [ ]:
try:
    import matplotlib
    import matplotlib.pyplot as plt

    # Sort by predicted median for a cleaner plot
    sort_idx = np.argsort(pred_50)
    x_axis = np.arange(len(sort_idx))

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.fill_between(x_axis, pred_10[sort_idx], pred_90[sort_idx],
                    alpha=0.3, label="80% prediction interval (q10-q90)")
    ax.plot(x_axis, pred_50[sort_idx], label="Median prediction (q50)", linewidth=1.5)
    ax.scatter(x_axis, y_test[sort_idx], s=8, color="red", alpha=0.6, label="Actual values")
    ax.set_xlabel("Test samples (sorted by predicted median)")
    ax.set_ylabel("Target value")
    ax.set_title("EBM Quantile Regression: 80% Prediction Interval")
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed, skipping plot")

## Interpretability

One of the key advantages of quantile EBMs is that they remain fully interpretable. The global explanations show how each feature contributes to the predicted quantile, and local explanations show the additive score breakdown for individual predictions.

Let's compare the shape functions for the same feature across different quantiles.

In [ ]:
# Show global explanations for each quantile model
print("=== 10th Percentile Model ===")
show(ebm_10.explain_global())

In [ ]:
print("=== 90th Percentile Model ===")
show(ebm_90.explain_global())

In [ ]:
# Local explanation for a single test sample
show(ebm_50.explain_local(X_test[:5], y_test[:5]), 0)

## Making Predictions Manually

Just like standard regression EBMs, quantile EBM predictions are computed by summing the intercept with lookup table scores for each feature. The prediction logic is identical; only the training loss differs.

In [ ]:
# Use the small dataset to demonstrate manual predictions
X_small = [["Peru", 7.0], ["Fiji", 8.0], ["Peru", 9.0]]
y_small = [450.0, 550.0, 350.0]

sample_scores = []
for sample in X_small:
    score = ebm_median.intercept_
    print("intercept: " + str(score))

    for feature_idx, feature_val in enumerate(sample):
        bins = ebm_median.bins_[feature_idx][0]
        if isinstance(bins, dict):
            bin_idx = bins[feature_val]
        else:
            bin_idx = np.digitize(feature_val, bins) + 1

        local_score = ebm_median.term_scores_[feature_idx][bin_idx]
        print(ebm_median.feature_names_in_[feature_idx] + ": " + str(local_score))
        score += local_score
    sample_scores.append(score)
    print()

print("PREDICTIONS (manual):")
print(np.array(sample_scores))
print("PREDICTIONS (ebm.predict):")
print(ebm_median.predict(X_small))

## Summary

- Use `objective="quantile:alpha=0.5"` for median regression, or any alpha in (0, 1) for other quantiles.
- The prediction mechanism is identical to standard regression EBMs (intercept + additive score lookups). Only the training loss function changes.
- Fitting multiple quantile models (e.g. alpha=0.1 and alpha=0.9) provides interpretable prediction intervals.
- All EBM interpretability tools (global/local explanations) work with quantile models.